# Spaces tutorial

This notebook introduces the `Space` abstraction in `spacecore`.

It explains:

- what a space represents,
- how a space stores context,
- how a space encodes geometric structure,
- and how to use each currently implemented space.

Current implemented spaces:

- `VectorSpace`
- `HermitianSpace`
- `ProductSpace`


## 1. What a `Space` signifies

A `Space` is the library abstraction that represents a Hilbert space of numerical objects.
It encodes the geometric structure of its elements together with the operations that are natural on them.

In particular, a space does not only specify the ambient array shape. It also determines the structural properties and geometric operations associated with that class of objects.

Thus, a space may encode:

* shape,
* linear structure,
* inner product,
* norm,
* projection,
* structural constraints such as Hermitian symmetry,
* product structure.

So a `Space` captures the **geometry** of the underlying objects, not merely their storage format.

Schematically, the role split is

$$
\texttt{BackendOps} \to \texttt{Context} \to \texttt{Space} \to \texttt{LinOp}.
$$

* `BackendOps` says how numerical computations are performed.
* `Context` says which backend, dtype, and checking policy are active.
* `Space` says what kind of mathematical objects we are working with and which geometry they carry.

## 2. Every space stores a context

Each space carries a `Context`.

That means a space is not only a geometric object, but also a numerical one: it knows under which backend and dtype policy its elements live.

So if `X` is a space, then `X.ctx: Context` determines:

- which backend arrays are expected,
- which dtype is targeted,
- whether runtime validation checks are enabled.

This is why a space can do things like:

- create zero elements,
- test membership,
- add and scale elements,
- compute inner products,
- project objects back into the space,

all in a backend-aware way.


In [ ]:
import numpy as np

from spacecore.backend import Context, NumpyOps
from spacecore.space import VectorSpace, HermitianSpace, ProductSpace

ctx = Context(NumpyOps(), dtype=np.float64, enable_checks=True)

X = VectorSpace((3,), ctx=ctx)
print(X)
print(X.ctx)


## 3. Core ideas shared by all spaces

Although concrete spaces differ, they typically share the following ideas:

- they define what counts as a valid element,
- they provide basic linear operations,
- they define an inner product and norm when appropriate,
- they may provide geometry-specific projections,
- they store context and therefore know how to create and validate elements.

So a space can be viewed as a structured set

$$
X \subseteq \mathbb{F}^{n_1 \times \cdots \times n_k},
$$

together with geometric operations appropriate for that subset.


## 4. `VectorSpace`

`VectorSpace` is the basic dense linear space.

It represents ordinary arrays of a fixed shape, with the standard linear structure inherited from the backend.

So if

$$
X = \texttt{VectorSpace}(\texttt{shape}=(n_1,\dots,n_k)),
$$

then its elements are simply dense arrays of shape `(n_1, ..., n_k)` compatible with the stored context.

This is the most basic space in the library.


In [ ]:
ctx = Context(NumpyOps(), dtype=np.float64, enable_checks=True)
X = VectorSpace((2, 3), ctx=ctx)

x = X.zeros()
y = ctx.asarray([[1, 2, 3], [4, 5, 6]])

print('x =')
print(x)
print('y =')
print(y)
print('shape:', X.shape)


### 4.1 Basic linear operations in `VectorSpace`

A vector space should support the standard operations

$$
x + y, \qquad \alpha x, \qquad \langle x, y \rangle, \qquad \|x\|.
$$

The exact implementation is backend-dependent, but the geometric meaning is the familiar one.


In [ ]:
x = ctx.asarray([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
y = ctx.asarray([[0.0, 1.0, 0.0], [1.0, 0.0, 1.0]])

print('x + y =')
print(X.add(x, y))

print('2x =')
print(X.scale(2.0, x))

print('inner(x, y) =', X.inner(x, y))
print('norm(x) =', X.norm(x))


### 4.2 Membership in `VectorSpace`

Membership is primarily geometric and contextual:

- correct shape,
- correct backend representation,
- correct dtype policy when checks are enabled.

So `VectorSpace` is the space of admissible dense arrays of the given shape under the stored context.


In [ ]:
good = np.zeros((2, 3))
bad = np.zeros((3,))

X.check_member(good)
print('Good element passed membership check.')

try:
    X.check_member(bad)
except Exception as e:
    print('Bad element rejected:', type(e).__name__, e)


## 5. `HermitianSpace`

`HermitianSpace` is the space of Hermitian matrices of a fixed square shape.

So if

$$
H = \texttt{HermitianSpace}(n),
$$

then its elements are matrices

$$
A \in \mathbb{F}^{n \times n}
\qquad\text{such that}\qquad
A = A^*.
$$

This space is more structured than `VectorSpace`: it does not just encode shape, but also Hermitian symmetry.

Because of that, it can provide geometry-specific operations such as symmetrization, eigendecomposition, and projection onto the positive semidefinite cone.


In [ ]:
ctx_h = Context(NumpyOps(), dtype=np.complex128, enable_checks=True)
H = HermitianSpace(3, ctx=ctx_h)

A = ctx_h.asarray([
    [1, 1 + 2j, 0],
    [1 - 2j, 3, 4j],
    [0, -4j, 2],
])

print(A)
H.check_member(A)
print('Hermitian matrix accepted.')


### 5.1 Symmetrization

Given a general square matrix \(M\), the Hermitian part is

$$
\frac{M + M^*}{2}.
$$

This is the canonical projection of a square matrix onto the linear subspace of Hermitian matrices.


In [ ]:
M = ctx_h.asarray([
    [1, 2 + 1j, 0],
    [0, 3, 5],
    [7j, 1, 2],
])

M_herm = H.symmetrize(M)
print(M_herm)
H.check_member(M_herm)
print('Symmetrized matrix is Hermitian.')


### 5.2 Eigendecomposition

A Hermitian matrix admits an eigendecomposition

$$
A = U \operatorname{diag}(\lambda) U^*.
$$

This is fundamental in Hermitian geometry and is one reason why `HermitianSpace` is useful as a dedicated abstraction.


In [ ]:
evals, evecs = H.eigh(A)
print('eigenvalues =', evals)
print('eigenvectors =')
print(evecs)


### 5.3 Projection onto the PSD cone

For Hermitian matrices, a common geometric operation is projection onto the cone of positive semidefinite matrices.

This amounts to clipping negative eigenvalues:

$$
A = U \operatorname{diag}(\lambda) U^*
\quad\mapsto\quad
U \operatorname{diag}(\max(\lambda,0)) U^*.
$$

This operation is naturally attached to `HermitianSpace`, not to a generic vector space.


In [ ]:
B = ctx_h.asarray([
    [1, 2, 0],
    [2, -3, 0],
    [0, 0, 1],
])
B = H.symmetrize(B)

B_psd = H.psd_proj(B)
print(B_psd)


## 6. `ProductSpace`

`ProductSpace` represents a Cartesian product of spaces.

So if

$$
X = X_1 \times \cdots \times X_k,
$$

then an element of `ProductSpace((X1, ..., Xk))` is a tuple

$$
(x_1, \dots, x_k)
\qquad\text{with } x_i \in X_i.
$$

This is useful whenever a variable is naturally block-structured.

Importantly, `ProductSpace` is not only a container of spaces. It defines the geometry of tuples of space elements and provides structured operations such as flattening and unflattening.


In [ ]:
X1 = VectorSpace((2,), ctx=ctx)
X2 = VectorSpace((3,), ctx=ctx)
X_prod = ProductSpace((X1, X2), ctx=ctx)

x = (ctx.asarray([1.0, 2.0]), ctx.asarray([3.0, 4.0, 5.0]))
print(X_prod)
print(x)


### 6.1 Block structure

The geometry of a product space is tuple geometry.

For example, addition and scaling are performed componentwise:

$$
(x_1,\dots,x_k) + (y_1,\dots,y_k)
= (x_1+y_1,\dots,x_k+y_k),
$$

$$
\alpha (x_1,\dots,x_k)
= (\alpha x_1,\dots,\alpha x_k).
$$

This is exactly the right geometry for block variables.


In [ ]:
y = (ctx.asarray([10.0, 20.0]), ctx.asarray([30.0, 40.0, 50.0]))

print('x + y =')
print(X_prod.add(x, y))

print('2x =')
print(X_prod.scale(2.0, x))

print('inner(x, y) =', X_prod.inner(x, y))


### 6.2 Flattening and unflattening

A particularly useful operation for product spaces is to move between tuple form and a flat vector representation.

If

$$
x = (x_1,\dots,x_k) \in X_1 \times \cdots \times X_k,
$$

then flattening produces one vector containing all blocks, and unflattening reconstructs the tuple.

This is often essential when connecting structured variables with linear operators or optimization routines.


In [ ]:
flat = X_prod.flatten(x)
print('flattened =', flat)

x_back = X_prod.unflatten(flat)
print('unflattened =', x_back)


## 7. Context inside product spaces

A `ProductSpace` also stores a context.

In the current design, product spaces are homogeneous in context: component spaces are converted into the product-space context.

So the product space represents a structured tuple geometry under one common numerical policy.


## 8. Choosing the right space

A simple rule is:

- use `VectorSpace` for generic dense arrays,
- use `HermitianSpace` when Hermitian structure matters,
- use `ProductSpace` when your variable is naturally block-structured.

So the choice of space should follow the geometry of your problem, not merely the raw storage layout.


## 9. Why the space abstraction matters

Without spaces, all code would directly manipulate raw arrays.
Then geometric meaning would remain implicit.

With spaces, the library can make geometry explicit.
For example:

- Hermitian structure belongs to `HermitianSpace`,
- block structure belongs to `ProductSpace`,
- standard linear geometry belongs to `VectorSpace`.

This improves correctness, readability, and backend independence.


## 10. Summary

A `Space` is the abstraction that represents the geometry of admissible numerical objects.

It stores a context, so every space is also numerically grounded in a backend and dtype policy.

Current implemented spaces are:

- `VectorSpace`: generic dense arrays of fixed shape,
- `HermitianSpace`: Hermitian matrices of fixed square shape,
- `ProductSpace`: tuples of elements from component spaces.

So the role of spaces is to turn raw arrays into explicit geometric objects inside the library.
